In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Libraries

In [28]:
import numpy as np
import pandas as pd
import string
import nltk

!pip install gensim

from nltk.stem import PorterStemmer , WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.ensemble import AdaBoostClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer

import gensim.downloader as api
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt_tab')
from gensim.models import Word2Vec # Import the Word2Vec class



from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Read dataset

In [29]:
filepath = '/content/drive/MyDrive/AI ML Course/Data/DonorsChoose.csv'
df_donor = pd.read_csv(filepath)
df_donor.head(8)

,id,teacher_prefix,school_state,project_grade_category,project_subject_categories,project_subject_subcategories,teacher_number_of_previously_posted_projects,project_is_approved,price,quantity,cleaned_titles,cleaned_essays,cleaned_summary,isdigit_summary
0,p253737,mrs,in,grades_prek_2,literacy_language,esl_literacy,0,0,154.60,23,educational support english learners home,students english learners working english seco...,students_need_opportunities_practice_beginning...,0
1,p258326,mr,fl,grades_6_8,history_civics_health_sports,civics_government_teamsports,7,1,299.00,1,wanted projector hungry learners,students arrive school eager learn polite gene...,students_need_projector_help_viewing_education...,0
2,p182444,ms,az,grades_6_8,health_sports,health_wellness_teamsports,1,0,516.85,22,soccer equipment awesome middle school students,true champions not always ones win guts mia ha...,students_need_shine_guards_athletic_socks_socc...,0
3,p246581,mrs,ky,grades_prek_2,literacy_language_math_science,literacy_mathematics,4,1,232.90,4,techie kindergarteners,work unique school filled esl english second l...,students_need_engage_reading_math_way_inspire_...,0
4,p104768,mrs,tx,grades_prek_2,math_science,mathematics,1,1,67.98,4,interactive math tools,second grade classroom next year made around 2...,students_need_hands_practice_mathematics_fun_p...,0
5,p154343,mrs,fl,grades_3_5,literacy_language_specialneeds,literature_writing_specialneeds,1,1,113.22,11,flexible seating mrs jarvis terrific third gra...,moving 2nd grade 3rd grade beginning next scho...,students_need_movement_successful_variety_stud...,0
6,p099819,mrs,ct,grades_6_8,literacy_language_specialneeds,literacy_specialneeds,1,1,159.99,3,chromebooks special education reading program,students dynamic energetic group middle school...,students_need_dependable_laptops_daily_classro...,0
7,p092424,ms,ga,grades_3_5,math_science,mathematics,7,1,229.00,4,21st century,not students struggle poverty also learning ma...,students_need_ipads_help_access_world_online_r...,0


## EDA

In [30]:
df_donor.shape

(109248, 14)

In [31]:
df_donor.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109248 entries, 0 to 109247
Data columns (total 14 columns):
 #   Column                                        Non-Null Count   Dtype  
---  ------                                        --------------   -----  
 0   id                                            109248 non-null  object 
 1   teacher_prefix                                109248 non-null  object 
 2   school_state                                  109248 non-null  object 
 3   project_grade_category                        109248 non-null  object 
 4   project_subject_categories                    109248 non-null  object 
 5   project_subject_subcategories                 109248 non-null  object 
 6   teacher_number_of_previously_posted_projects  109248 non-null  int64  
 7   project_is_approved                           109248 non-null  int64  
 8   price                                         109248 non-null  float64
 9   quantity                                      10

In [32]:
df_donor.describe()

,teacher_number_of_previously_posted_projects,project_is_approved,price,quantity,isdigit_summary
count,109248.000000,109248.000000,109248.000000,109248.000000,109248.000000
mean,11.153165,0.848583,298.119343,16.965610,0.144222
std,27.777154,0.358456,367.498030,26.182942,0.351317
min,0.000000,0.000000,0.660000,1.000000,0.000000
25%,0.000000,1.000000,104.310000,4.000000,0.000000
50%,2.000000,1.000000,206.220000,9.000000,0.000000
75%,9.000000,1.000000,379.000000,21.000000,0.000000
max,451.000000,1.000000,9999.000000,930.000000,1.000000


In [33]:
df_donor.isnull().sum()

,0
id,0
teacher_prefix,0
school_state,0
project_grade_category,0
project_subject_categories,0
project_subject_subcategories,0
teacher_number_of_previously_posted_projects,0
project_is_approved,0
price,0
quantity,0


## Preprocessing

### Missing value handling

In [34]:
df_donor['cleaned_titles'] = df_donor['cleaned_titles'].fillna('No Title')
df_donor.isna().sum()

,0
id,0
teacher_prefix,0
school_state,0
project_grade_category,0
project_subject_categories,0
project_subject_subcategories,0
teacher_number_of_previously_posted_projects,0
project_is_approved,0
price,0
quantity,0


### Duplicates handling


In [35]:
df_donor.duplicated().sum()

np.int64(0)

### our target

In [36]:
df_donor['project_is_approved'].unique()
df_donor['project_is_approved'].value_counts()

,count
project_is_approved,
1,92706
0,16542


In [37]:
# the data is imbalanced

df_donor['cleaned_titles'].head()

,cleaned_titles
0,educational support english learners home
1,wanted projector hungry learners
2,soccer equipment awesome middle school students
3,techie kindergarteners
4,interactive math tools


In [38]:
df_donor['text'] = (
    df_donor['cleaned_titles'] + " " +
    df_donor['cleaned_summary'] + " " +
    df_donor['cleaned_essays']
)
df_donor[['text']].head()

,text
0,educational support english learners home stud...
1,wanted projector hungry learners students_need...
2,soccer equipment awesome middle school student...
3,techie kindergarteners students_need_engage_re...
4,interactive math tools students_need_hands_pra...


### Tokenization

In [39]:
df_donor['tokens'] = df_donor['text'].apply(word_tokenize)
df_donor[['text', 'tokens']].head()

,text,tokens
0,educational support english learners home stud...,"[educational, support, english, learners, home..."
1,wanted projector hungry learners students_need...,"[wanted, projector, hungry, learners, students..."
2,soccer equipment awesome middle school student...,"[soccer, equipment, awesome, middle, school, s..."
3,techie kindergarteners students_need_engage_re...,"[techie, kindergarteners, students_need_engage..."
4,interactive math tools students_need_hands_pra...,"[interactive, math, tools, students_need_hands..."


## TF-IDF Vectorization

In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_tfidf = tfidf.fit_transform(df_donor['text'])

print(X_tfidf.shape)

(109248, 5000)


## Split the Data

In [41]:
from sklearn.model_selection import train_test_split

X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(
    X_tfidf,
    df_donor['project_is_approved'],
    test_size=0.2,
    random_state=42,
    stratify=df_donor['project_is_approved']
)

## Word2Vec

In [ ]:
from gensim.models import Word2Vec

sentences = df_donor['tokens']

w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

In [ ]:
def document_vector(doc):

    words = [word for word in doc if word in w2v_model.wv]

    if len(words)==0:
        return np.zeros(100)

    return np.mean(w2v_model.wv[words],axis=0)

X_w2v = np.array(df_donor['tokens'].apply(document_vector).tolist())

print(X_w2v.shape)

## Split Word2Vec Data

In [ ]:
X_train_w2v, X_test_w2v, y_train_w2v, y_test_w2v = train_test_split(
    X_w2v,
    df_donor['project_is_approved'],
    test_size=0.2,
    random_state=42,
    stratify=df_donor['project_is_approved']
)

## Logistic Regression (TF-IDF)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

lr = LogisticRegression(max_iter=1000)

lr.fit(X_train_tfidf,y_train)

pred_lr = lr.predict(X_test_tfidf)

print("Accuracy :",accuracy_score(y_test,pred_lr))

print(classification_report(y_test,pred_lr))

print(confusion_matrix(y_test,pred_lr))

## Support Vector Machine

In [ ]:
from sklearn.svm import LinearSVC

svm = LinearSVC()

svm.fit(X_train_tfidf,y_train)

pred_svm = svm.predict(X_test_tfidf)

print("Accuracy :",accuracy_score(y_test,pred_svm))

print(classification_report(y_test,pred_svm))

print(confusion_matrix(y_test,pred_svm))

##  Random Forest Classifier (Word2Vec)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train_w2v, y_train_w2v)

pred_rf = rf.predict(X_test_w2v)

print("Accuracy:", accuracy_score(y_test_w2v, pred_rf))
print(classification_report(y_test_w2v, pred_rf))
print(confusion_matrix(y_test_w2v, pred_rf))

## Cross Validation

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    LogisticRegression(max_iter=1000),
    X_train_tfidf,
    y_train,
    cv=5,
    scoring='accuracy'
)

print("Cross Validation Scores:")
print(cv_scores)

print("Average Accuracy:", cv_scores.mean())

## Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

parameters = {
    'C':[0.01,0.1,1,10]
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    parameters,
    cv=5,
    scoring='accuracy'
)

grid.fit(X_train_tfidf, y_train)

print("Best Parameters:",grid.best_params_)
print("Best Score:",grid.best_score_)

## Convert TF-IDF to Dense Matrix

In [ ]:
X_train_tfidf_array = X_train_tfidf.toarray()
X_test_tfidf_array = X_test_tfidf.toarray()

## Prepare Word2Vec Train/Test Data

In [ ]:
from nltk.tokenize import word_tokenize
import numpy as np

train_tokens = X_train.apply(word_tokenize)
test_tokens = X_test.apply(word_tokenize)

def document_vector(doc):

    words = [word for word in doc if word in w2v_model.wv]

    if len(words)==0:
        return np.zeros(100)

    return np.mean(w2v_model.wv[words], axis=0)

X_train_w2v = np.array(train_tokens.apply(document_vector).tolist())
X_test_w2v = np.array(test_tokens.apply(document_vector).tolist())

## Build Functional API Model

In [ ]:
from tensorflow.keras.layers import Input,Dense,Dropout,Concatenate
from tensorflow.keras.models import Model

tfidf_input = Input(shape=(5000,),name='TFIDF_Input')

x1 = Dense(256,activation='relu')(tfidf_input)
x1 = Dropout(0.3)(x1)

w2v_input = Input(shape=(100,),name='Word2Vec_Input')

x2 = Dense(64,activation='relu')(w2v_input)
x2 = Dropout(0.3)(x2)

merged = Concatenate()([x1,x2])

merged = Dense(128,activation='relu')(merged)
merged = Dropout(0.3)(merged)

output = Dense(1,activation='sigmoid')(merged)

model = Model(
    inputs=[tfidf_input,w2v_input],
    outputs=output
)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Train the Model

In [ ]:
history = model.fit(
    [X_train_tfidf_array,X_train_w2v],
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64
)

## Model Prediction

In [ ]:
pred = model.predict(
    [X_test_tfidf_array,X_test_w2v]
)

pred = (pred>0.5).astype(int)

print(classification_report(y_test,pred))

## Model Evaluation

### Confusion Matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(y_test,pred)

plt.title("Confusion Matrix")
plt.show()

### ROC Curve

In [ ]:
from sklearn.metrics import roc_curve,auc

prob = model.predict(
    [X_test_tfidf_array,X_test_w2v]
)

fpr,tpr,_ = roc_curve(y_test,prob)

roc_auc = auc(fpr,tpr)

plt.figure(figsize=(6,5))
plt.plot(fpr,tpr,label="AUC = %.3f"%roc_auc)
plt.plot([0,1],[0,1],'--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

### Precision–Recall Curve

In [ ]:
from sklearn.metrics import precision_recall_curve

precision,recall,_ = precision_recall_curve(y_test,prob)

plt.figure(figsize=(6,5))
plt.plot(recall,precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

## Model Comparison & Analysis

In [ ]:

comparison = pd.DataFrame({
    "Model":["Logistic Regression","SVM","Random Forest","Functional API"],
    "Accuracy":[
        accuracy_score(y_test,pred_lr),
        accuracy_score(y_test,pred_svm),
        accuracy_score(y_test_w2v,pred_rf),
        accuracy_score(y_test,pred)
    ],
    "Precision":[
        precision_score(y_test,pred_lr),
        precision_score(y_test,pred_svm),
        precision_score(y_test_w2v,pred_rf),
        precision_score(y_test,pred)
    ],
    "Recall":[
        recall_score(y_test,pred_lr),
        recall_score(y_test,pred_svm),
        recall_score(y_test_w2v,pred_rf),
        recall_score(y_test,pred)
    ],
    "F1-Score":[
        f1_score(y_test,pred_lr),
        f1_score(y_test,pred_svm),
        f1_score(y_test_w2v,pred_rf),
        f1_score(y_test,pred)
    ]
})

comparison